In [5]:
!pip install openai-whisper transformers torch pandas soundfile

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 17.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=d7391700c5067c4ed75b2263478a9779f54ceb402abc0bf2c585851f8cb14918
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [6]:
import whisper
from transformers import pipeline
import pandas as pd
import torch
import os

device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {'GPU (cuda)' if device == 0 else 'CPU'}")

whisper_model = whisper.load_model("base")
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=device)
print("\nAll models loaded successfully!")

Using device: GPU (cuda)


100%|███████████████████████████████████████| 139M/139M [00:01<00:00, 89.2MiB/s]
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]


All models loaded successfully!


In [22]:
def process_smartnode_call(audio_path, fallback_text=None):
    if fallback_text and not os.path.exists(audio_path):
        transcript = fallback_text
        print(f"\n[Simulation Mode] Processing: {transcript[:50]}...")
    else:
        audio_result = whisper_model.transcribe(audio_path, fp16=False)
        transcript = audio_result["text"]
    candidate_labels = ["Resolved and completed", "Active issue or follow-up needed", "Urgent emergency fault"]

    classification_result = classifier(transcript, candidate_labels)
    label_mapping = {
        "Resolved and completed": "Closed",
        "Active issue or follow-up needed": "Open",
        "Urgent emergency fault": "Urgent"
    }

    top_label_descriptive = classification_result['labels'][0]
    top_category = label_mapping[top_label_descriptive]
    confidence_score = classification_result['scores'][0]

    return {
        "Transcript": transcript,
        "Assigned Category": top_category,
        "Confidence Score": round(confidence_score, 4),
        "Full Breakdown": dict(zip(classification_result['labels'], classification_result['scores']))
    }

In [23]:
all_test_scenarios = [
    {
        "id": "Case_01 (Urgent Fault)",
        "text": "Mera smart panel completely dead ho gaya hai, switches short circuit ho rahe hain drop points pe! Please check kijiye emergency hai lights respond nahi kar rahi."
    },
    {
        "id": "Case_02 (Open Loop)",
        "text": "Hello, technical team came yesterday to fix my Wi-Fi configuration pairing but the app keeps disconnecting repeatedly. The previous ticket is still unresolved, updates?"
    },
    {
        "id": "Case_03 (Resolution w/ Query)",
        "text": "Thank you, the automation gateway configuration reset worked perfectly. Everything is working fine now, I just wanted to confirm the warranty timeline on my 8-channel module."
    },
    {
        "id": "Case_04 (Pure Resolution)",
        "text": "Thank you so much for fixing the smart switch. Everything is working perfectly now and I don't have any other questions. Great service!"
    }
]

final_results = []
for case in all_test_scenarios:
    output = process_smartnode_call(audio_path="", fallback_text=case['text'])
    final_results.append({
        "Case ID": case['id'],
        "Assigned Category": output["Assigned Category"],
        "Confidence": output["Confidence Score"],
        "Transcript": output["Transcript"]
    })

df_final_metrics = pd.DataFrame(final_results)
display(df_final_metrics)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



[Simulation Mode] Processing: Mera smart panel completely dead ho gaya hai, swit...

[Simulation Mode] Processing: Hello, technical team came yesterday to fix my Wi-...

[Simulation Mode] Processing: Thank you, the automation gateway configuration re...

[Simulation Mode] Processing: Thank you so much for fixing the smart switch. Eve...


,Case ID,Assigned Category,Confidence,Transcript
0,Case_01 (Urgent Fault),Urgent,0.7052,"Mera smart panel completely dead ho gaya hai, ..."
1,Case_02 (Open Loop),Open,0.9858,"Hello, technical team came yesterday to fix my..."
2,Case_03 (Resolution w/ Query),Closed,0.8556,"Thank you, the automation gateway configuratio..."
3,Case_04 (Pure Resolution),Closed,0.7923,Thank you so much for fixing the smart switch....


### Test with your own Audio File
Run the cell below to upload a recording (e.g., .wav or .mp3) and see how the system classifies it.

In [29]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

if uploaded:
    all_results = []
    for filename in uploaded.keys():
        print(f"Processing: {filename}...")

        result = process_smartnode_call(audio_path=filename)

        all_results.append({
            "File": filename,
            "Transcript": result['Transcript'],
            "Category": result['Assigned Category'],
            "Confidence": result['Confidence Score']
        })

    print("\n" + "="*20 + " SUMMARY REPORT " + "="*20)
    df_summary = pd.DataFrame(all_results)
    display(df_summary)

Saving Vrajdham Haveli Marg 20.m4a to Vrajdham Haveli Marg 20 (1).m4a
Saving Vrajdham Haveli Marg 19.m4a to Vrajdham Haveli Marg 19 (1).m4a
Saving Vrajdham Haveli Marg 18.m4a to Vrajdham Haveli Marg 18 (1).m4a
Saving Vrajdham Haveli Marg 17.m4a to Vrajdham Haveli Marg 17 (1).m4a
Saving Vrajdham Haveli Marg 16.m4a to Vrajdham Haveli Marg 16 (1).m4a
Saving Vrajdham Haveli Marg 15.m4a to Vrajdham Haveli Marg 15 (1).m4a
Saving Vrajdham Haveli Marg 11.m4a to Vrajdham Haveli Marg 11 (1).m4a
Saving Vrajdham Haveli Marg 12.m4a to Vrajdham Haveli Marg 12 (1).m4a
Saving Vrajdham Haveli Marg 13.m4a to Vrajdham Haveli Marg 13 (1).m4a
Saving Vrajdham Haveli Marg 14.m4a to Vrajdham Haveli Marg 14 (1).m4a
Saving Vrajdham Haveli Marg 8.m4a to Vrajdham Haveli Marg 8 (2).m4a
Saving Vrajdham Haveli Marg 6.m4a to Vrajdham Haveli Marg 6 (2).m4a
Saving Vrajdham Haveli Marg 7.m4a to Vrajdham Haveli Marg 7 (2).m4a
Saving Vrajdham Haveli Marg 5.m4a to Vrajdham Haveli Marg 5 (2).m4a
Saving Vrajdham Haveli Marg 

,File,Transcript,Category,Confidence
0,Vrajdham Haveli Marg 20 (1).m4a,kya today my speech board will be good this i...,Open,0.8444
1,Vrajdham Haveli Marg 19 (1).m4a,제가 ipci sweating,Open,0.8223
2,Vrajdham Haveli Marg 18 (1).m4a,guyou can see it today my switched apps go on...,Open,0.7148
3,Vrajdham Haveli Marg 17 (1).m4a,"Alo, aap Nagיתar For the funical",Open,0.6701
4,Vrajdham Haveli Marg 16 (1).m4a,V dimensional wifi configurationzam支ی ederان ...,Open,0.7010
5,Vrajdham Haveli Marg 15 (1).m4a,"It is very urgent, I want to fix my router co...",Open,0.5299
6,Vrajdham Haveli Marg 11 (1).m4a,It is very emergency situation for me because...,Urgent,0.6918
7,Vrajdham Haveli Marg 12 (1).m4a,I have already come to know about this situat...,Closed,0.6310
8,Vrajdham Haveli Marg 13 (1).m4a,Please fix my life 5 configuration. Today end...,Open,0.8770
9,Vrajdham Haveli Marg 14 (1).m4a,It is emergency and I want to fix my Wi-Fi an...,Urgent,0.6619


In [30]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=df_summary)

https://docs.google.com/spreadsheets/d/1fCTfHCePx5_CxRT8EMlgZeXOQBxgWFqoZ2UFNxVvZZk/edit#gid=0


In [17]:
display(df_metrics)

,Case ID,Assigned Category,Confidence,Extracted Text
0,Case_01 (Urgent Fault),Urgent,0.9620,"Mera smart panel completely dead ho gaya hai, ..."
1,Case_02 (Open Loop / Follow-up),Urgent,0.5071,"Hello, technical team came yesterday to fix my..."
2,Case_03 (Closed / Resolution),Open,0.5449,"Thank you, the automation gateway configuratio..."
3,Case_04 (Pure Resolution),Open,0.4994,Thank you so much for fixing the smart switch....


In [13]:
case_03_data = df_metrics[df_metrics['Case ID'].str.contains('Case_03')]
display(case_03_data)

,Case ID,Assigned Category,Confidence,Extracted Text
2,Case_03 (Closed / Resolution),Open,0.5449,"Thank you, the automation gateway configuratio..."
